# R04 · Cookbook Reference Lab: NumPy batch API and layouts

**NHyDB 0.28.2 notebook series · v1 · reference part**

These examples are based on the official
[NHyDB Python Cookbook](https://github.com/nhydb-run4/NHyDB/blob/main/docs/cookbook/PYTHON_COOKBOOK.md) (sections 16–17), which serves as their authoritative reference.
Code and expected output are taken unchanged from the cookbook, and the same examples are run by the
NHyDB test suite.

* Every code cell is an **independent example** – run any cell on its own, or *Run All*.
* Each example creates its own temporary database folder.
* Under each example you find the output printed in the cookbook, to compare with yours.
* The main series (`../main/`) explains the ideas behind these APIs step by step.

**Contents** (6 runnable examples)

* 16. NumPy batch API
* 17. Layouts: COLUMN, PACKED, CUSTOM, routing and evolution

In [1]:
import nhydb

print('nhydb', nhydb.ENGINE_VERSION, '· file format', nhydb.FILE_FORMAT_VERSION)
if nhydb.ENGINE_VERSION != '0.28.2':
    print('⚠ These examples were written for NHyDB 0.28.2; results may differ.')

nhydb 0.28.2 · file format 0.28.2


## 16. NumPy batch API

The NumPy methods copy whole columns into the engine; encoding and the
write loops run in Rust. All examples in this section need NumPy.

In [2]:
# requires: numpy
import os
import tempfile

import numpy as np

import nhydb

path = os.path.join(tempfile.mkdtemp(), "ts.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "measurements")
    db.create_shape(1, 1, "sample", [
        {"v": 1, "name": "t", "type": "u64"},
        {"v": 2, "name": "temp", "type": "f32"},
        {"v": 3, "name": "ok", "type": "bool"},
        {"v": 4, "name": "station", "type": "utf8"},
    ])
    db.create_classic_layout(1, 1, "samples", shape_id=1)

    n = 100_000
    t = np.arange(n, dtype=np.uint64) * 60
    cols = {
        "t": t,
        "temp": (20 + np.sin(t / 3600.0)).astype(np.float32),
        "ok": t % 7 != 0,
        "station": np.array(["A", "B"] * (n // 2), dtype=object),
    }
    db.put_numpy_batch(1, 1, cols, commit=True)        # records u = 1 … 100000

    out = db.read_numpy(1, 99_999, 3, ["t", "temp", "station"])
    print(out["t"], out["temp"].dtype, list(out["station"]), out["_present"])

[5999880 5999940       0] float32 ['A', 'B', None] [ True  True False]


*Expected output (from the cookbook):*

```text
[5999880 5999940       0] float32 ['A', 'B', None] [ True  True False]
```

`read_numpy` returns one array per field plus `_present`; missing records
have `_present=False` (values 0, NaN or `None`). Integer types come back as
`int64` (`u64` and `varuint` as `uint64`), `utf8`/`bytes`/enums as object arrays. The
dtype depends only on the field type; a stored value that does not fit
(for example a `varuint` above 2^64 − 1) raises `ValueError` unless you pass
`object_fallback=True`. A `count` too large for the memory raises
`MemoryError`.

Objects of any layout, sparse cells and query results:

In [3]:
# requires: numpy
import os
import tempfile

import numpy as np

import nhydb
from nhydb.query import Q, QueryEngine

path = os.path.join(tempfile.mkdtemp(), "np2.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    db.create_shape(1, 1, "s", [{"v": 1, "name": "id", "type": "varuint"}, {"v": 2, "name": "x", "type": "f64"}])
    db.create_classic_layout(1, 1, "points", shape_id=1)

    ids = np.array([10, 11, 12, 50], dtype=np.uint64)
    db.put_objects_numpy(1, ids, {"id": ids, "x": ids / 4})                  # any layout, by object id
    db.put_cells_numpy(1, np.full(3, 500, dtype=np.uint64), np.array([1, 2, 3], dtype=np.uint64),
                       np.array([1.5, 2.5, 3.5]), type="f64", lane="NR")   # sparse cells
    db.commit()

    print(db.get_record(1, 50)["x"], db.get_value(1, 500, 3, lane="NR"))
    print(db.sql_numpy("SELECT id, x FROM points WHERE id < ? ORDER BY id", [50]))
    r = QueryEngine(db).execute_numpy(Q.field("x") > 2.8, realm=1, select=["id"])
    print(r["id"], QueryEngine(db).execute_sql_numpy("SELECT x FROM points WHERE id = 11")["x"])

12.5 3.5
{'id': array([10, 11, 12]), 'x': array([2.5 , 2.75, 3.  ])}
[12 50] [2.75]


*Expected output (from the cookbook):*

```text
12.5 3.5
{'id': array([10, 11, 12]), 'x': array([2.5 , 2.75, 3.  ])}
[12 50] [2.75]
```

Accepted column types: integer arrays of any width and byte order (ranges
are checked) for integer fields, float or integer arrays for `f32`/`f64`,
bool or integer arrays for `bool`, object arrays or lists for text, bytes
and enum labels (integer arrays for enum codes). Columns must be
one-dimensional and of equal length; bool or float arrays for integer
fields, numeric arrays for text fields and dtypes such as float16, complex
or datetime raise `TypeError`. `u`/`v`/object-id arrays are limited to
`uint64`; use lists of Python `int` for larger coordinates; a scalar `u`/`v`
is broadcast to all rows. Invalid input raises `TypeError`, `ValueError`
(wrong length, out of range) or `SchemaError` (missing field) before
anything is written. A batch becomes one block; it is read and checked
completely once on first use, after that single accesses read only the
bytes they need, so large batches stay fast to read (10,000–100,000
records per batch remain a good default). `read_numpy`/`sql_numpy` return
writable arrays that share the engine's result memory (no extra copy).
`put_math_numpy` is shown in section 18.

## 17. Layouts: COLUMN, PACKED, CUSTOM, routing and evolution

| Layout | Record id | Field | Use it for |
|---|---|---|---|
| CLASSIC | `u` | `v` (shape field) | ordinary tables (sections 3–8) |
| COLUMN | `v` | one fixed `u` per field | column-oriented data: all values of one field share a `u` |
| PACKED | `⌊(v−1)/span⌋ + 1` | offset in a block of `span` cells | many small records packed along `v` of one `u` |
| CUSTOM | your plugin | your plugin | any mapping you implement in Python |

`put_object`, `put_objects_batch`, `get_object` and `delete_record` work
for every layout; `put_record`/`get_record` are the same calls with
simpler names.

In [4]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "layouts.nhydb")
with nhydb.create(path) as db:
    db.create_realm(2, "columns")
    db.create_column_layout(10, 2, "metrics", fields=[
        {"u": 1, "name": "temp", "type": "f64"},
        {"u": 2, "name": "humidity", "type": "u8"},
    ])
    db.put_object(10, 42, {"temp": 21.5, "humidity": 40})
    print(db.get_object(10, 42))
    print(db.get_object(10, 42, include_addresses=True)["_addresses"]["temp"]["u"])

    db.create_realm(3, "packed")
    db.create_packed_layout(11, 3, "pairs", u=1, span=8, fields=[
        {"offset": 0, "name": "a", "type": "varuint"},
        {"offset": 1, "name": "b", "type": "utf8"},
    ])
    print(db.put_objects_batch(11, [(1, {"a": 1, "b": "x"}), (2, {"a": 2, "b": "y"})])["objects"])
    anchor = db.record_anchor(11, 2)
    print(anchor["u"], anchor["v_local"])            # record 2 starts at v = (2-1)*8 + 1
    print(db.delete_record(11, 1), db.get_object(11, 1), db.get_object(11, 2)["b"])
    db.commit()

{'_layout': 10, '_layout_name': 'metrics', '_kind': 'COLUMN', '_object': 42, '_lane': 0, '_lane_name': 'NL', 'temp': 21.5, 'humidity': 40}
1
2
1 9
2 None y


*Expected output (from the cookbook):*

```text
{'_layout': 10, '_layout_name': 'metrics', '_kind': 'COLUMN', '_object': 42, '_lane': 0, '_lane_name': 'NL', 'temp': 21.5, 'humidity': 40}
1
2
1 9
2 None y
```

**Several layouts in one realm (routing).** Layouts can share a realm with
different `u` ranges. For a given `u` the layout with the higher
`priority` wins, then the narrower range, then the higher id.

In [5]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "routing.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "mixed")
    db.create_shape(1, 1, "s", [{"v": 1, "name": "x", "type": "u32"}])
    db.create_classic_layout(1, 1, "all", shape_id=1)                          # u = 1 … ∞
    db.create_layout(2, 1, "special", "CLASSIC", shape_id=1, u_start=100, u_end=199, priority=5)

    print(db.layout_at(1, 50).name, db.layout_at(1, 150).name)
    print([l.name for l in db.layout_candidates(1, 150)])
    print(db.route_address(1, 150, 1)["layout_name"])
    print(db.layout_diagnostics(1)["summary"])
    print(db.get_layout(layout_id=2)["u_range"])
    lay = db.get_layout(layout_id=2)
    print(db.layout_at(1, 150).to_dict()["priority"], db.layout_at(1, 150).contains_u(150))

all special
['special', 'all']
special
{'realms': 1, 'overlaps': 1, 'conflicts': 0, 'gaps': 0}
{'from': 100, 'to': 199}
5 True


*Expected output (from the cookbook):*

```text
all special
['special', 'all']
special
{'realms': 1, 'overlaps': 1, 'conflicts': 0, 'gaps': 0}
{'from': 100, 'to': 199}
5 True
```

**Changing a layout safely.** `plan_layout_evolution` shows the effect of a
change without applying it; `evolve_layout` applies it. Changes that would
give already stored data a different layout owner are rejected with
`SchemaError`.

In [6]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "evolve.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    db.create_shape(1, 1, "s", [{"v": 1, "name": "x", "type": "u32"}])
    db.create_layout(1, 1, "first", "CLASSIC", shape_id=1, u_start=1, u_end=1000)
    db.put(1, 5, [7], commit=True)

    plan = db.plan_layout_evolution(1, u_end=None)
    print(plan["status"], plan["routing_changes"][0]["from"])
    print(db.evolve_layout(1, u_end=None, description="open ended", commit=True))
    try:
        db.evolve_layout(1, u_start=10)                  # would orphan the record at u=5
    except nhydb.SchemaError:
        print("rejected")

safe 1001
LayoutDef(layout_id=1, realm_id=1, name="first", kind='CLASSIC', u_start=1, u_end=None, priority=0)
rejected


*Expected output (from the cookbook):*

```text
safe 1001
LayoutDef(layout_id=1, realm_id=1, name="first", kind='CLASSIC', u_start=1, u_end=None, priority=0)
rejected
```

`evolve_layout(..., add_fields=[...])` adds fields to a COLUMN/PACKED/CUSTOM
layout; `spacemap_update` applies to CUSTOM layouts only.

**CUSTOM layouts** delegate the mapping to a Python object that you
register at runtime (only the plugin name is stored). The plugin needs
`logical_address(layout, object_id, field, lane)` and, for reading,
`type_of(layout, address)` and `describe(layout, address)`. Optional:
`spacemap` and `occupied_addresses(db, layout, object_id, lane)`.

In [7]:
import os
import tempfile

import nhydb


class KeyValuePairs:
    """Object k stores 'key' at v = 2k-1 and 'val' at v = 2k, all in u = 100."""

    def logical_address(self, layout, object_id, field, lane):
        offset = {None: 0, "key": 0, "val": 1}[field]
        return {"u": 100, "v": (object_id - 1) * 2 + offset + 1, "lane": lane}

    def type_of(self, layout, address):
        return "utf8"

    def describe(self, layout, address):
        v = address["v_local"]
        return {"record": (v - 1) // 2 + 1, "field": ("key", "val")[(v - 1) % 2], "anchor": (v - 1) % 2 == 0}


path = os.path.join(tempfile.mkdtemp(), "custom.nhydb")
with nhydb.create(path) as db:
    db.create_realm(50, "kv")
    db.create_custom_layout(9, 50, "pairs", plugin="kv-pairs", u_start=100, u_end=100,
                            params={"fields": [{"name": "key"}, {"name": "val"}]})
    db.register_layout_plugin("kv-pairs", KeyValuePairs())   # needed again after every open
    db.put_object(9, 3, {"key": "color", "val": "blue"}, commit=True)
    print(db.get_object(9, 3)["val"], db.logical_address(9, 3, "val")["v_local"])
    print(db.get_n(db.logical_address(9, 3, "key")["N"])["data"])

blue 6
{'key': 'color', 'val': 'blue'}


*Expected output (from the cookbook):*

```text
blue 6
{'key': 'color', 'val': 'blue'}
```

Without a registered plugin, operations that need the mapping raise
`SchemaError("CUSTOM layout requires registered plugin '…'")`. A plugin
must not write into the same database from inside a callback.
`occupied_addresses(db, layout, object_id, lane)` is also called during
`delete_record`, while `db` is locked (`RuntimeError: Already mutably
borrowed`); compute the addresses from `layout`, `object_id` and `lane`
only and do not call `db` there.

`create_layout(...)` is the generic form of all `create_*_layout` methods
(`kind="CLASSIC" | "COLUMN" | "PACKED" | "CUSTOM"`, `replace=True` to
redefine a layout whose address space is still empty).